---
title: "IPCC AR6 Gender Distribution"
format:
  html:
    code-fold: true
execute:
  enabled: false
---

# IPCC AR6 Gender Distribution

A simple visualization showing the gender distribution of all authors in the IPCC AR6 Assessment Report.

**Instructions:**
1. Update the SPARQL_ENDPOINT variable below with your production endpoint
2. Run all cells in this notebook to generate the chart
3. The output will be saved in the notebook
4. Quarto will render the stored output without re-executing

**Data Source:** SPARQL endpoint (Wikibase)  
**Visualization:** Single Plotly bar chart

In [24]:
#| include: false
# Import required libraries
import sys
import socket
import requests
import pandas as pd
import plotly.express as px
from SPARQLWrapper import SPARQLWrapper, JSON

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## Configuration

Update `WIKI_BASE_URL` and `SPARQL_ENDPOINT` to match your environment:

| Environment | Wiki Base URL | Query UI | SPARQL Endpoint |
|-------------|---------------|----------|-----------------|
| **LOCAL** | `http://localhost:8080` | `http://localhost:8081` | `http://localhost:9999/bigdata/namespace/wdq/sparql` |
| **DEV** | `https://dev-climatekg.semanticclimate.org` | `https://dev-climatekg.semanticclimate.org/query/` | `https://dev-climatekg.semanticclimate.org/query/proxy/sparql` |
| **TEST** | `https://test-climatekg.semanticclimate.org` | `https://test-climatekg.semanticclimate.org/query/` | `https://test-climatekg.semanticclimate.org/query/proxy/sparql` |
| **PROD** | `https://prod-climatekg.semanticclimate.org` | `https://prod-climatekg.semanticclimate.org/query/` | `https://prod-climatekg.semanticclimate.org/query/proxy/sparql` |

**Note:** The SPARQL query prefixes (`wdt:`, `wd:`) are built from `WIKI_BASE_URL` automatically.

In [25]:
#| include: false
# ============================================================
# CONFIGURATION - Update these two variables for your environment
# ============================================================

# Wiki base URL (no trailing slash)
WIKI_BASE_URL = "http://localhost:8080"

# SPARQL endpoint URL
SPARQL_ENDPOINT = "http://localhost:9999/bigdata/namespace/wdq/sparql"

# For other environments, use these values:
# DEV:
#   WIKI_BASE_URL = "https://dev-climatekg.semanticclimate.org"
#   SPARQL_ENDPOINT = "https://dev-climatekg.semanticclimate.org/query/proxy/sparql"
#
# TEST:
#   WIKI_BASE_URL = "https://test-climatekg.semanticclimate.org"
#   SPARQL_ENDPOINT = "https://test-climatekg.semanticclimate.org/query/proxy/sparql"
#
# PROD:
#   WIKI_BASE_URL = "https://prod-climatekg.semanticclimate.org"
#   SPARQL_ENDPOINT = "https://prod-climatekg.semanticclimate.org/query/proxy/sparql"

# ============================================================

print(f"Wiki Base URL:    {WIKI_BASE_URL}")
print(f"SPARQL Endpoint:  {SPARQL_ENDPOINT}")
print(f"\nPREFIX wdt: <{WIKI_BASE_URL}/prop/direct/>")
print(f"PREFIX wd:  <{WIKI_BASE_URL}/entity/>")

Wiki Base URL:    http://localhost:8080
SPARQL Endpoint:  http://localhost:9999/bigdata/namespace/wdq/sparql

PREFIX wdt: <http://localhost:8080/prop/direct/>
PREFIX wd:  <http://localhost:8080/entity/>


## Check SPARQL Endpoint Availability

**Troubleshooting if connection fails:**

1. **Is the WDQS container running?**
   - `docker compose ps` — all 5 containers should show `healthy` or `running`
   - `docker compose up -d` to start the stack

2. **Correct LOCAL URLs:**
   - Wiki: `http://localhost:8080`
   - Query UI (web interface): `http://localhost:8081`
   - SPARQL endpoint (direct access): `http://localhost:9999/bigdata/namespace/wdq/sparql`

3. **Test the endpoint in your browser:**
   - Open http://localhost:9999/bigdata/namespace/wdq/sparql

In [26]:
#| include: false
# Quick diagnostic - Check what's running on localhost
ports_to_check = {
    8080: "MediaWiki/Wikibase (Wiki)",
    8081: "Query UI",
    9999: "WDQS/Blazegraph SPARQL",
}

print("Checking localhost ports...\n")
for port, service in ports_to_check.items():
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(1)
    result = sock.connect_ex(('localhost', port))
    sock.close()
    
    if result == 0:
        print(f"✓ Port {port} is OPEN  - {service}")
    else:
        print(f"✗ Port {port} is CLOSED - {service}")

print("\nIf port 9999 is closed, start the stack: docker compose up -d")

Checking localhost ports...

✓ Port 8080 is OPEN  - MediaWiki/Wikibase (Wiki)
✓ Port 8081 is OPEN  - Query UI
✓ Port 9999 is OPEN  - WDQS/Blazegraph SPARQL

If port 9999 is closed, start the stack: docker compose up -d


In [27]:
#| include: false
# Test SPARQL endpoint connectivity
print("Testing SPARQL endpoint connectivity...")
print(f"Target: {SPARQL_ENDPOINT}\n")

try:
    # Try a simple HEAD request first (faster)
    print("Step 1: Checking if endpoint is reachable...")
    response = requests.head(SPARQL_ENDPOINT, timeout=10)
    
    if response.status_code in [200, 405]:  # 405 is OK (endpoint doesn't support HEAD)
        print(f"✓ SPARQL endpoint is reachable\n")
        
        # Test with a minimal query to verify it actually works
        print("Step 2: Testing with a simple query...")
        test_query = "SELECT * WHERE { ?s ?p ?o } LIMIT 1"
        test_response = requests.post(
            SPARQL_ENDPOINT,
            data={'query': test_query},
            headers={'Accept': 'application/sparql-results+json'},
            timeout=20
        )
        
        if test_response.status_code == 200:
            print(f"✓ SPARQL endpoint is responding to queries\n")
            print("✓ Connection test PASSED - Ready to query data!")
        else:
            print(f"✗ Warning: Endpoint reachable but query failed (HTTP {test_response.status_code})")
            print("The endpoint may not be configured correctly.")
            print("\nTry accessing the endpoint in your browser to verify it works:")
            print(f"  {SPARQL_ENDPOINT}")
            sys.exit(1)
    else:
        print(f"✗ Error: SPARQL endpoint returned HTTP {response.status_code}")
        print(f"✗ Cannot connect to: {SPARQL_ENDPOINT}")
        print("\nPlease check:")
        print("1. Blazegraph is running (not just MediaWiki)")
        print("2. The SPARQL endpoint URL is correct")
        print("3. Network/firewall settings allow the connection")
        sys.exit(1)
        
except requests.exceptions.ConnectionError:
    print(f"✗ Error: Cannot connect to SPARQL endpoint")
    print(f"✗ URL: {SPARQL_ENDPOINT}")
    print("\nDiagnosis: Port is not accepting connections")
    print("\nPlease check:")
    print("1. Is Blazegraph running? (Run the diagnostic cell above to check ports)")
    print("2. If using Docker: docker ps | grep -i blazegraph")
    print("3. If using Docker Compose: docker-compose ps")
    print("4. Try starting services: docker-compose up -d")
    sys.exit(1)
    
except requests.exceptions.Timeout:
    print(f"✗ Error: Connection timeout (waited 10 seconds)")
    print(f"✗ URL: {SPARQL_ENDPOINT}")
    print("\nDiagnosis: Port is open but service not responding")
    print("\nPossible causes:")
    print("1. Blazegraph is starting up (may take 30-60 seconds after docker start)")
    print("2. Blazegraph is overloaded or crashed")
    print("3. Check Docker logs: docker logs <blazegraph-container-name>")
    print("\nTry waiting a minute and re-running this cell")
    sys.exit(1)
    
except Exception as e:
    print(f"✗ Unexpected error testing endpoint: {e}")
    print("\nIf you see SSL or certificate errors, your endpoint might be HTTPS")
    print("Update SPARQL_ENDPOINT to use https:// instead of http://")
    sys.exit(1)


Testing SPARQL endpoint connectivity...
Target: http://localhost:9999/bigdata/namespace/wdq/sparql

Step 1: Checking if endpoint is reachable...
✓ SPARQL endpoint is reachable

Step 2: Testing with a simple query...
✓ SPARQL endpoint is responding to queries

✓ Connection test PASSED - Ready to query data!


## Query Author Data from Wikibase

In [28]:
#| echo: false
# SPARQL query to get all authors with gender information
# PREFIX URIs are constructed from WIKI_BASE_URL set in configuration above
author_query = f"""
PREFIX wdt: <{WIKI_BASE_URL}/prop/direct/>
PREFIX wd: <{WIKI_BASE_URL}/entity/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?author ?authorLabel ?climateKGAuthorID ?gender
WHERE {{
    # Get all authors (instance of Q3998)
    ?author wdt:P1 wd:Q3998 .

    # Get author label
    OPTIONAL {{
        ?author rdfs:label ?authorLabel .
        FILTER(LANG(?authorLabel) = "en")
    }}

    # Get ClimateKG Author ID
    OPTIONAL {{
        ?author wdt:P20 ?climateKGAuthorID .
    }}

    # Get gender (stored as literal "M" or "F")
    OPTIONAL {{
        ?author wdt:P23 ?gender .
    }}
}}
ORDER BY ?climateKGAuthorID
"""

# Execute SPARQL query
print("Querying Wikibase for author data...")
print(f"Using Wiki base: {WIKI_BASE_URL}")
print(f"Using SPARQL endpoint: {SPARQL_ENDPOINT}")
print(f"Properties: P1 (instance of), P23 (gender), P20 (ClimateKG Author ID)\n")

sparql = SPARQLWrapper(SPARQL_ENDPOINT)
sparql.setQuery(author_query)
sparql.setReturnFormat(JSON)

try:
    results = sparql.query().convert()
    bindings = results["results"]["bindings"]

    # Convert to pandas DataFrame
    data = []
    for binding in bindings:
        row = {
            'author': binding['author']['value'],
            'authorLabel': binding.get('authorLabel', {}).get('value', ''),
            'climateKGAuthorID': binding.get('climateKGAuthorID', {}).get('value', 'N/A'),
            'gender': binding.get('gender', {}).get('value', 'Unknown')
        }
        data.append(row)

    df = pd.DataFrame(data)

    # Deduplicate by ClimateKG Author ID
    df_with_uid = df[df['climateKGAuthorID'] != 'N/A']
    df_unique = df_with_uid.drop_duplicates(subset='climateKGAuthorID')

    print(f"Raw SPARQL results:            {len(df):,}")
    print(f"  — with ClimateKG UID:        {len(df_with_uid):,}")
    print(f"  — without ClimateKG UID:     {(df['climateKGAuthorID'] == 'N/A').sum():,}")
    print(f"After dedup by UID:            {len(df_unique):,} unique authors")
    print(f"  — duplicates removed:        {len(df_with_uid) - len(df_unique):,}")
    print(f"  — with gender set:           {(df_unique['gender'] != 'Unknown').sum():,}")
    print(f"\nRaw gender values (pre-dedup):")
    print(df['gender'].value_counts().to_string())
    print(f"\nGender values (unique authors):")
    print(df_unique['gender'].value_counts().to_string())

except Exception as e:
    print(f"Error querying SPARQL endpoint: {e}")
    raise


Querying Wikibase for author data...
Using Wiki base: http://localhost:8080
Using SPARQL endpoint: http://localhost:9999/bigdata/namespace/wdq/sparql
Properties: P1 (instance of), P23 (gender), P20 (ClimateKG Author ID)

Raw SPARQL results:            1,864
  — with ClimateKG UID:        1,864
  — without ClimateKG UID:     0
After dedup by UID:            932 unique authors
  — duplicates removed:        932
  — with gender set:           932

Raw gender values (pre-dedup):
gender
M    1270
F     594

Gender values (unique authors):
gender
M    635
F    297


## Process Gender Data

In [29]:
#| echo: false
from IPython.display import display, HTML

# Map raw "M"/"F" codes to readable labels
gender_label_map = {'M': 'Male', 'F': 'Female', 'Unknown': 'Unknown'}
df['gender_label'] = df['gender'].map(gender_label_map).fillna('Unknown')

# Deduplicate by ClimateKG Author ID (keep unique authors only)
df_unique = df[df['climateKGAuthorID'] != 'N/A'].drop_duplicates(subset='climateKGAuthorID')

# Count gender distribution on unique authors
gender_counts = df_unique['gender_label'].value_counts().reset_index()
gender_counts.columns = ['Gender', 'Count']
gender_counts = gender_counts.sort_values('Gender')

# Calculate percentages
total = gender_counts['Count'].sum()
gender_counts['Percentage'] = (gender_counts['Count'] / total * 100).round(1)

print("Gender Distribution (unique authors by ClimateKG Author ID):")
print("=" * 45)
print(gender_counts.to_string(index=False))
print(f"\nTotal unique authors: {total:,}")

# Create bar chart
fig = px.bar(
    gender_counts,
    x='Gender',
    y='Count',
    title='IPCC AR6 Authors: Gender Distribution',
    text='Count',
    color='Gender',
    color_discrete_map={
        'Male': '#2E86AB',
        'Female': '#A23B72',
        'Unknown': '#6C757D'
    }
)

# Add percentage labels on bars
fig.update_traces(
    texttemplate='%{text}<br>(%{customdata:.1f}%)',
    textposition='outside',
    customdata=gender_counts['Percentage']
)

fig.update_layout(
    height=500,
    showlegend=False,
    xaxis_title='Gender',
    yaxis_title='Number of Authors',
    font=dict(size=12),
    yaxis=dict(range=[0, gender_counts['Count'].max() * 1.15])
)

# Render as self-contained HTML so Quarto embeds it without any external dependencies
display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))


Gender Distribution (unique authors by ClimateKG Author ID):
Gender  Count  Percentage
Female    297        31.9
  Male    635        68.1

Total unique authors: 932
